# Data science: longitudinal phenotypes → mixed model → prediction

**For:** data science, critical care research, quantitative science, research data analysts.

**Questions:**
1. Can I build an analysis-ready person × timepoint dataset from many source tables?
2. How do depressive symptoms (PHQ-9) evolve over 4 years, and does that differ by depression history?
3. From enrollment data alone, how well can we predict new-onset moderate depressive symptoms
   (PHQ-9 ≥ 10) in years 3–4?

**Requirements:** a workspace with the Project Baseline Health Study (Limited Tier) data collection added (resources `screener`, `crf`, `analysis`, `externallab`, `corelabreads`, `appsurveys`).

**Data:** Baseline (Limited Tier) app surveys (PHQ-9 twice a year, PSS, PROMIS), CRF (GAD-7, vitals),
labs, and derived history flags. Adults only. *Illustrative modeling, not a validated clinical model.*

In [ ]:
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from google.cloud import bigquery
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss

bq = bigquery.Client()

_resolved = {}

def tbl(dataset, table):
    """Look up a workspace dataset by resource name: tbl('crf', 'MH') -> `<project>.crf.MH`."""
    if dataset not in _resolved:
        _resolved[dataset] = subprocess.run(["wb", "resolve", "--name", dataset],
                                            capture_output=True, text=True, check=True).stdout.strip()
    return f"`{_resolved[dataset]}.{table}`"

def sql(q):
    return bq.query(q).to_dataframe()

# VISITNUM: 1 = enrollment, 2 = year 1, ... ; decimals .1/.2/.3 = quarterly app surveys
YEARS = "(FLOOR(VISITNUM) - 1) + (VISITNUM - FLOOR(VISITNUM)) * 2.5"

## 1. Long-format PHQ-9 trajectories

In [ ]:
phq = sql(f"""
SELECT USUBJID, {YEARS} AS years, SAFE_CAST(PHQ9_SUM_SCORE AS FLOAT64) AS phq9
FROM {tbl('analysis', 'PHQ9A_SCORES')} WHERE VISITNUM IS NOT NULL AND PHQ9_SUM_SCORE IS NOT NULL
""")
hx = sql(f"""
SELECT USUBJID, mh_major_depression AS hx_depression, mh_sleepapnea AS hx_sleep_apnea, mh_diabetes AS hx_diabetes,
       mh_htn AS hx_htn, mh_copd AS hx_copd
FROM {tbl('analysis', 'DIAGNOSES')} WHERE VISIT = 'STUDY_START'
""")
demo = sql(f"SELECT USUBJID, SAFE_CAST(age_at_enrollment AS INT64) AS age, SEX AS sex, RACE AS race FROM {tbl('screener', 'DM')}")
hx = hx.set_index("USUBJID").astype(float).reset_index()      # BigQuery nullable ints -> plain floats
demo["age"] = demo.age.astype(float)
phq = phq.merge(hx, on="USUBJID").merge(demo, on="USUBJID")
print(f"{len(phq):,} PHQ-9 measurements · {phq.USUBJID.nunique():,} participants · "
      f"median {phq.groupby('USUBJID').size().median():.0f} per person")

fig, ax = plt.subplots(figsize=(8, 4))
for flag, color, label in [(0, "#3b6ea5", "no depression history"), (1, "#c0504d", "depression history")]:
    m = phq[phq.hx_depression == flag].groupby("years").phq9.agg(["mean", "sem"])
    ax.errorbar(m.index, m["mean"], yerr=1.96 * m["sem"], color=color, marker="o", capsize=3, label=label)
ax.axhline(10, color="grey", ls=":", lw=1); ax.text(0, 10.2, "moderate (≥10)", fontsize=8, color="grey")
ax.set_xlabel("years since enrollment"); ax.set_ylabel("mean PHQ-9 (95% CI)"); ax.legend()
ax.set_title("PHQ-9 over 4 years"); ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

## 2. Linear mixed model (random intercept per participant)

In [ ]:
mm = smf.mixedlm("phq9 ~ years * hx_depression + age + C(sex)", phq.dropna(subset=["age", "hx_depression"]),
                 groups="USUBJID").fit()
mm.summary().tables[1]

## 3. Analysis-ready enrollment feature table
One row per participant, built from 7 source tables in 3 datasets.

In [ ]:
features = sql(f"""
WITH
gad AS (SELECT USUBJID, gad7_1_CODE + gad7_2_CODE + gad7_3_CODE + gad7_4_CODE + gad7_5_CODE + gad7_6_CODE + gad7_7_CODE AS gad7
        FROM {tbl('crf', 'GAD7')} WHERE VISIT = 'STUDY_START'),
pss AS (SELECT USUBJID, AVG(PSS_GLOBAL_SCORE) AS pss FROM {tbl('analysis', 'PSS_SCORES')} WHERE VISITNUM < 2 GROUP BY 1),
promis AS (SELECT USUBJID, AVG(PROMIS_INTERFERENCE_SCORE) AS pain_interference
           FROM {tbl('analysis', 'PROMIS_SCORES')} WHERE VISITNUM < 2 GROUP BY 1),
vs AS (SELECT USUBJID, SAFE_DIVIDE(SAFE_CAST(vs_weight_kg AS FLOAT64), POW(SAFE_CAST(vs_height_cm AS FLOAT64)/100, 2)) AS bmi,
              SAFE_CAST(vs_sbp1_mmhg AS FLOAT64) AS sbp
       FROM {tbl('crf', 'VS')} WHERE VISIT = 'STUDY_START'),
labs AS (SELECT USUBJID,
           AVG(IF(rtcname = 'CRP HIGH SENSITIVITY', SAFE_CAST(siurslt AS FLOAT64), NULL)) AS hs_crp,
           AVG(IF(rtcname = 'HEMOGLOBIN A1C', SAFE_CAST(siurslt AS FLOAT64), NULL)) AS a1c
         FROM {tbl('externallab', 'CLABS')} WHERE VISIT = 'STUDY_START' GROUP BY 1)
SELECT * FROM gad FULL JOIN pss USING (USUBJID) FULL JOIN promis USING (USUBJID)
         FULL JOIN vs USING (USUBJID) FULL JOIN labs USING (USUBJID)
""").drop_duplicates("USUBJID")

baseline_phq = phq[phq.years < 1].groupby("USUBJID").phq9.mean().rename("phq9_baseline")
outcome = (phq[phq.years >= 3].groupby("USUBJID").phq9.max() >= 10).astype(int).rename("phq9_ge10_y3_4")

data = (demo.merge(hx, on="USUBJID").merge(features, on="USUBJID", how="left")
            .merge(baseline_phq, on="USUBJID").merge(outcome, on="USUBJID"))
data = data[data.phq9_baseline < 10]            # at risk: not already moderate at enrollment
data["female"] = (data.sex == "Female").astype(int)
data["non_white"] = (data.race != "White").astype(int)
FEATURES = ["age", "female", "non_white", "phq9_baseline", "gad7", "pss", "pain_interference", "bmi", "sbp",
            "hs_crp", "a1c", "hx_depression", "hx_sleep_apnea", "hx_diabetes", "hx_htn", "hx_copd"]
X, y = data[FEATURES].astype(float), data.phq9_ge10_y3_4
print(f"At-risk cohort: {len(data):,} · new-onset PHQ-9 ≥10 in years 3–4: {y.sum()} ({y.mean():.1%})")
data[FEATURES + ["phq9_ge10_y3_4"]].describe().T[["count", "mean", "50%"]].round(2)

## 4. Cross-validated prediction: logistic regression vs. gradient boosting

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)
models = {
    "Logistic regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                         LogisticRegression(max_iter=1000)),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
fig, ax = plt.subplots(figsize=(5, 5))
results = {}
for name, m in models.items():
    p = cross_val_predict(m, X, y, cv=cv, method="predict_proba")[:, 1]
    results[name] = {"AUROC": roc_auc_score(y, p), "Brier": brier_score_loss(y, p)}
    fpr, tpr, _ = roc_curve(y, p)
    ax.plot(fpr, tpr, label=f"{name} (AUROC {results[name]['AUROC']:.2f})")
ax.plot([0, 1], [0, 1], color="grey", ls=":")
ax.set_xlabel("1 − specificity"); ax.set_ylabel("sensitivity"); ax.legend(fontsize=8)
ax.set_title("5-fold cross-validated ROC"); plt.tight_layout(); plt.show()
pd.DataFrame(results).T.round(3)

In [ ]:
lr = models["Logistic regression"].fit(X, y)
odds = pd.Series(np.exp(lr[-1].coef_[0]), index=FEATURES).sort_values()
ax = odds.plot.barh(figsize=(6, 5), color=np.where(odds > 1, "#c0504d", "#3b6ea5"))
ax.axvline(1, color="grey", lw=1); ax.set_xlabel("odds ratio per 1 SD (standardized)")
ax.set_title("Logistic regression: enrollment predictors"); plt.tight_layout(); plt.show()

**Reproducibility:** save `data` (the analysis-ready table) and this notebook to a workspace bucket so
collaborators can rerun it. Every number here traces back to the SQL above.

**Try next:** add wearable-derived or sleep features (`appsurveys.SH`), swap the outcome
(incident prediabetes from A1c, rising hs-CRP), or use temporal validation (train on early enrollees).

## Provenance

Generate information about this notebook environment and the packages installed.

In [ ]:
!date

Conda and pip installed packages:

In [ ]:
!conda env export

JupyterLab extensions:

In [ ]:
!jupyter labextension list

Number of cores:

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

Memory:

In [ ]:
!grep "^MemTotal:" /proc/meminfo

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd